# 02 — Training (native bf16, torch.compile, W&B, resume-safe)

Edit the **CONFIG** cell to pick the model and run. Checkpoints (`last.pth`/`best.pth`) and the
W&B run-id persist to `PERSIST_DIR`, so re-running after a session ends **resumes** automatically —
the key to spanning the three 6-hour GPU windows.

Suggested runs: Day 1 `resnet50` then `vit_b`; Day 2 `vit_l` (hero), plus `vit_l` with
`use_stain_aug=False` for the ablation, plus `phikon`.

In [1]:
# Locate shared.ipynb regardless of the kernel's working directory, then run it.
from pathlib import Path as _P
_sh = next((p for p in [_P.cwd() / "shared.ipynb",
                        _P("/workspace/shared/ft004/shared.ipynb")] if p.exists()), None)
if _sh is None:
    _hits = list(_P.cwd().rglob("shared.ipynb")) or list(_P("/workspace").rglob("shared.ipynb"))
    _sh = _hits[0] if _hits else None
assert _sh, "shared.ipynb not found - keep it beside the notebooks or in /workspace/shared/ft004"
print("running", _sh)
get_ipython().run_line_magic("run", str(_sh))
import os
from pathlib import Path
PERSIST_DIR = Path(os.environ.get("PERSIST_DIR", "/workspace/shared/ft004"))
set_seed(42)


running /workspace/finetuning-004-pathology/shared.ipynb


shared.ipynb loaded: Phase A + B + helpers ready.


In [2]:
# ===================== CONFIG =====================
cfg = dict(
    model="phikon",           # resnet50 | vit_b | vit_l | swin_b | phikon
    use_stain_aug=True,       # set False for the ablation counterpart
    hed_theta=0.05,
    img_size=224,
    epochs=12,
    batch_size=256,           # ViT-L on MI300X can push 512; ResNet/ViT-B 256
    eval_batch_size=512,
    lr=1e-4,                  # head LR (backbone uses lr * backbone_lr_mult)
    backbone_lr_mult=0.1,
    weight_decay=0.05,
    warmup_epochs=1,
    label_smoothing=0.1,
    num_workers=8,            # needs container --ipc=host / --shm-size; lower if bus errors
    pin_memory=True,
    prefetch_factor=4,
    use_compile=True,         # torch.compile(reduce-overhead); falls back to eager on failure
    resume=True,
)
run_name = f"{cfg['model']}_{'stain' if cfg['use_stain_aug'] else 'nostain'}"
ckpt_dir = PERSIST_DIR / "checkpoints" / run_name
ckpt_dir.mkdir(parents=True, exist_ok=True)
device = get_device()
amp_dtype = get_amp_dtype(device)
print(f"run={run_name} device={device} amp={amp_dtype}")

run=phikon_stain device=cuda amp=torch.bfloat16


In [3]:
# Data
paths = ensure_dataset(PERSIST_DIR)
train_loader, val_loader, classes = make_dataloaders(
    paths["NCT-CRC-HE-100K"], paths["CRC-VAL-HE-7K"], cfg)
print(f"classes={classes}")
print(f"train batches={len(train_loader)} val batches={len(val_loader)}")


21:22:45 | INFO | NCT-CRC-HE-100K already present: /workspace/shared/ft004/data/NCT-CRC-HE-100K


21:22:45 | INFO | CRC-VAL-HE-7K already present: /workspace/shared/ft004/data/CRC-VAL-HE-7K/CRC-VAL-HE-7K


/tmp/ipykernel_40826/3600200050.py:33: UserWarning: Argument(s) 'mode' are not valid for transform Affine
  A.Affine(translate_percent=0.05, scale=(0.9, 1.1), rotate=(-15, 15),


21:22:49 | INFO | Dataset NCT-CRC-HE-100K: 100000 imgs / 9 classes


21:22:49 | INFO | Dataset CRC-VAL-HE-7K: 7180 imgs / 9 classes


classes=['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']
train batches=390 val batches=15


In [4]:
# Model, optimizer, scheduler
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

model = build_model(cfg, num_classes=len(classes)).to(device)
optimizer = torch.optim.AdamW(get_param_groups(model, cfg))
warmup = LinearLR(optimizer, start_factor=0.1, total_iters=max(1, cfg["warmup_epochs"]))
cosine = CosineAnnealingLR(optimizer, T_max=max(1, cfg["epochs"] - cfg["warmup_epochs"]),
                           eta_min=cfg["lr"] * 0.01)
scheduler = SequentialLR(optimizer, [warmup, cosine], milestones=[cfg["warmup_epochs"]])
criterion = nn.CrossEntropyLoss(label_smoothing=cfg["label_smoothing"])

# fp16 (non-MI300X) needs a GradScaler; bf16 on MI300X does NOT.
scaler = torch.cuda.amp.GradScaler() if amp_dtype == torch.float16 else None

if cfg["use_compile"] and device.type == "cuda":
    try:
        # "reduce-overhead" (CUDA graphs) corrupts BatchNorm running stats on ROCm,
        # causing immediate NaN loss for conv nets; use plain compile for those.
        has_bn = any(isinstance(m, nn.BatchNorm2d) for m in model.modules())
        compile_mode = "default" if has_bn else "reduce-overhead"
        model = torch.compile(model, mode=compile_mode)
        print(f"torch.compile: enabled ({compile_mode})")
    except Exception as e:
        print(f"torch.compile failed -> eager ({e})")


config.json:   0%|          | 0.00/492 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

torch.compile: enabled (reduce-overhead)


In [5]:
# Resume from last.pth if present (atomic checkpoints written every epoch)
import wandb
start_epoch, best_f1 = 0, 0.0
last_path = ckpt_dir / "last.pth"
if cfg["resume"] and last_path.exists():
    ck = load_checkpoint(last_path)
    raw_module(model).load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    start_epoch = ck["epoch"] + 1
    best_f1 = ck.get("best_metric", 0.0)
    logger.info(f"Resumed {run_name} from epoch {start_epoch} (best_f1={best_f1:.2f})")

# Persist W&B run id so a restarted session continues the SAME run
run_id_file = ckpt_dir / "wandb_run_id.txt"
run_id = run_id_file.read_text().strip() if run_id_file.exists() else wandb.util.generate_id()
run_id_file.write_text(run_id)
wandb.init(project="finetuning_004", id=run_id, resume="allow", name=run_name, config=cfg)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb: Currently logged in as: thunderhill4 (thunderhill4-tcs) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.27.2


wandb: Run data is saved locally in /workspace/finetuning-004-pathology/wandb/run-20260615_212257-3zzuzqef
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run phikon_stain


wandb: ⭐️ View project at https://wandb.ai/thunderhill4-tcs/finetuning_004


wandb: 🚀 View run at https://wandb.ai/thunderhill4-tcs/finetuning_004/runs/3zzuzqef


In [6]:
def train_one_epoch(model, loader, optimizer, criterion, device, amp_dtype, scaler, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    use_amp = device.type == "cuda"
    t0 = time.time()
    for step, (x, y) in enumerate(loader):
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                out = model(x)
                loss = criterion(out.float(), y)
        else:
            out = model(x); loss = criterion(out, y)
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); optimizer.step()
        total_loss += loss.item() * y.size(0); total += y.size(0)
        correct += (out.argmax(1) == y).sum().item()
        if step % 50 == 0:
            ips = total / max(1e-6, time.time() - t0)
            logger.info(f"ep{epoch} step{step}/{len(loader)} loss {loss.item():.4f} "
                        f"acc {100*correct/total:.1f}% {ips:.0f} img/s")
    return total_loss / total, 100 * correct / total


In [7]:
# Training loop — eval every epoch, log to W&B, save best + last (resume-safe)
for epoch in range(start_epoch, cfg["epochs"]):
    logger.info(f"===== epoch {epoch+1}/{cfg['epochs']} =====")
    try:
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion,
                                           device, amp_dtype, scaler, epoch)
        scheduler.step()
        preds, labels = run_inference(model, val_loader, device, amp_dtype)
        m = compute_metrics(labels, preds, classes)
        logger.info(f"val acc {m['accuracy']:.2f}% macro-F1 {m['macro_f1']:.2f}%")

        wandb.log({"epoch": epoch, "train/loss": tr_loss, "train/acc": tr_acc,
                   "val/acc": m["accuracy"], "val/macro_f1": m["macro_f1"],
                   "lr": optimizer.param_groups[0]["lr"],
                   **{f"val/f1_{c}": v for c, v in m["per_class_f1"].items()}})
        try:
            wandb.log({"val/confusion": wandb.plot.confusion_matrix(
                probs=None, y_true=labels.tolist(), preds=preds.tolist(),
                class_names=classes)})
        except Exception as e:
            logger.warning(f"wandb confusion log skipped ({e})")

        state = dict(epoch=epoch, model=raw_module(model).state_dict(),
                     optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                     best_metric=best_f1, metrics=m, cfg=cfg, classes=classes)
        save_checkpoint(state, last_path)
        if m["macro_f1"] > best_f1:
            best_f1 = m["macro_f1"]; state["best_metric"] = best_f1
            save_checkpoint(state, ckpt_dir / "best.pth")
            logger.info(f"** new best macro-F1 {best_f1:.2f}% -> best.pth **")
    except Exception as e:
        logger.exception(f"epoch {epoch} failed; last.pth retained for resume ({e})")
        raise

logger.info(f"DONE {run_name}: best macro-F1 {best_f1:.2f}%")
wandb.finish()


21:22:58 | INFO | ===== epoch 1/12 =====


21:23:57 | INFO | ep0 step0/390 loss 2.2391 acc 9.0% 4 img/s


21:24:01 | INFO | ep0 step50/390 loss 1.9978 acc 24.3% 208 img/s


21:24:08 | INFO | ep0 step100/390 loss 1.8206 acc 39.8% 369 img/s


21:24:15 | INFO | ep0 step150/390 loss 1.6371 acc 51.1% 504 img/s


...

wandb: uploading artifact run-3zzuzqef-valconfusion_table-LDzIYQ


wandb: uploading config.yaml; uploading media/table/val/confusion_table_23_44ac722ddb630de71b44.table.json; uploading wandb-summary.json


wandb: 
wandb: Run history:
wandb:       epoch ▁▂▂▃▄▄▅▅▆▇▇█
wandb:          lr ██▇▇▆▅▄▃▂▂▁▁
wandb:   train/acc ▁▇██████████
wandb:  train/loss █▂▁▁▁▁▁▁▁▁▁▁
wandb:     val/acc ▁▆▆▆▇▇▇█████
wandb:  val/f1_ADI █▆▂▁▆▃▅▆█▇▇█
wandb: val/f1_BACK ▁███████████
wandb:  val/f1_DEB ▁▆▇█▇▇██████
wandb:  val/f1_LYM ▁▄▆▇▇█▇█████
wandb:  val/f1_MUC ▁▅▇▇▇▆▆▆█▇▆▇
wandb:          +5 ...
wandb: 
wandb: Run summary:
wandb:       epoch 11
wandb:          lr 0.0
wandb:   train/acc 98.36238
wandb:  train/loss 0.55077
wandb:     val/acc 96.62953
wandb:  val/f1_ADI 99.4363
wandb: val/f1_BACK 100
wandb:  val/f1_DEB 98.9781
wandb:  val/f1_LYM 99.76322
wandb:  val/f1_MUC 99.61427
wandb:          +5 ...
wandb: 


wandb: 🚀 View run phikon_stain at: https://wandb.ai/thunderhill4-tcs/finetuning_004/runs/3zzuzqef
wandb: ⭐️ View project at: https://wandb.ai/thunderhill4-tcs/finetuning_004
wandb: Synced 4 W&B file(s), 12 media file(s), 24 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20260615_212257-3zzuzqef/logs
